In [1]:
# CONCLUSION FROM EDA HERE

In [2]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import resample
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

In [3]:
# Load Data
df = pd.read_csv('../../data/creditcard.csv') 

In [4]:
# Stratified split
raw_cols = ["Time", "Amount"] + [f"V{i}" for i in range(1,29)]
X = df[raw_cols]
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [5]:
# Building Pipeline
def secs_to_hour_of_day(x):
    '''Takes in raw seconds and returns the hour of day'''
    return (x % 86400) / 3600

time_transformer = FunctionTransformer(secs_to_hour_of_day, feature_names_out='one-to-one') 
#note: lambda doesn't work well with pickle
amount_transformer = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

column_transformations = ColumnTransformer(transformers=(
    ('time_to_hour_of_day', time_transformer,['Time']),
    ('amount_to_log', amount_transformer,['Amount'])),
    remainder = "passthrough"
    )

model = Pipeline([
    ('column_transformations', column_transformations),
    ('isolation_forest', IsolationForest(n_estimators=100, contamination=0.0017, random_state=42))
])
